# Notebook 01 — Exploración del Dataset Maps

> **Blog Técnico** · Pix2Pix: Traducción Bidireccional Satelital/Boceto

## El Dataset: Maps de Berkeley

El dataset proviene del repositorio original de Pix2Pix ([Isola et al., 2017](https://arxiv.org/abs/1611.07004)). Contiene **pares de imágenes** aerial/mapa capturadas sobre ciudades de EE.UU.

**Formato side-by-side**: cada archivo es una imagen de `1200×600 px` donde:
- **Mitad izquierda** (columnas 0–599): fotografía aérea/satelital
- **Mitad derecha** (columnas 600–1199): mapa de carreteras (OpenStreetMap style)

```
┌─────────────────┬─────────────────┐
│   Satelital     │   Mapa OSM      │
│   (dominio A)   │   (dominio B)   │
│   600 × 600 px  │   600 × 600 px  │
└─────────────────┴─────────────────┘
         Total: 1200 × 600 px
```

**Split:**
- `train/`: 1 096 pares
- `val/`:   1 098 pares

> El modelo aprende a traducir A→B (satelite a mapa) o B→A (mapa a satélite).

In [ ]:
import sys, os
from pathlib import Path

# Asegurar que el directorio raíz del proyecto está en el path
PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

from src.data.dataset_loader import DatasetParesSideBySide, crear_dataloader

DIR_DATOS = Path('data/processed')
print(f"Directorio de datos: {DIR_DATOS.resolve()}")
print(f"train/ existe: {(DIR_DATOS / 'train').exists()}")
print(f"val/   existe: {(DIR_DATOS / 'val').exists()}")

## 1. Estadísticas del Dataset

In [ ]:
from PIL import Image as PILImage
import os

def estadisticas_dir(ruta_dir):
    """Recopila estadísticas básicas de un directorio de imágenes."""
    imagenes = sorted(Path(ruta_dir).glob('*.jpg'))
    if not imagenes:
        return {}
    # Muestrear 20 imágenes para calcular tamaños
    muestra = imagenes[:20]
    anchos, altos = [], []
    for p in muestra:
        with PILImage.open(p) as img:
            anchos.append(img.width)
            altos.append(img.height)
    tam_disco_mb = sum(p.stat().st_size for p in imagenes) / 1e6
    return {
        'n': len(imagenes),
        'ancho_medio': int(np.mean(anchos)),
        'alto_medio': int(np.mean(altos)),
        'disco_mb': tam_disco_mb,
    }

print("=" * 50)
print("  Estadísticas del dataset Maps")
print("=" * 50)
for modo in ['train', 'val']:
    stats = estadisticas_dir(DIR_DATOS / modo)
    if stats:
        print(f"  {modo:6s}: {stats['n']:5d} pares | "
              f"{stats['ancho_medio']}×{stats['alto_medio']} px | "
              f"{stats['disco_mb']:.0f} MB en disco")
print("=" * 50)

## 2. Visualizar Pares Originales (Formato Raw)

Antes de pasar por el DataLoader (sin transformaciones), mostramos las imágenes tal como están en disco.

In [ ]:
imagenes_train = sorted((DIR_DATOS / 'train').glob('*.jpg'))
N_MOSTRAR = 4

fig, axes = plt.subplots(N_MOSTRAR, 2, figsize=(10, 3 * N_MOSTRAR))
fig.suptitle('Pares del Dataset Maps (raw, sin transformar)', fontsize=14, y=1.01)

for i, ruta in enumerate(imagenes_train[:N_MOSTRAR]):
    with PILImage.open(ruta) as img:
        ancho_medio = img.width // 2
        satelital = img.crop((0, 0, ancho_medio, img.height))
        mapa_osm  = img.crop((ancho_medio, 0, img.width, img.height))

    axes[i, 0].imshow(satelital)
    axes[i, 0].set_title(f'Dominio A — Satelital', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mapa_osm)
    axes[i, 1].set_title(f'Dominio B — Mapa OSM', fontsize=9)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('results/exploracion_pares_raw.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Figura guardada en results/exploracion_pares_raw.png")

## 3. Pares tras el Pipeline del DataLoader

El `DatasetParesSideBySide` aplica las siguientes transformaciones **sincronizadas** (mismo crop y flip para A y B):

1. `Resize(286)` — redimensionar al tamaño ligeramente mayor
2. `RandomCrop(256)` — recorte aleatorio a 256×256
3. `RandomHorizontalFlip()` — volteo horizontal con p=0.5 (solo en train)
4. `ToTensor()` — convertir PIL Image a tensor `float32 [0,1]`
5. `Normalize(mean=0.5, std=0.5)` — mapear `[0,1] → [-1,1]`

La normalización a **[-1, 1]** es imprescindible: la función de activación final del generador es `Tanh`, cuyo rango de salida es precisamente `[-1, 1]`.

In [ ]:
from src.data.dataset_loader import desnormalizar

# Crear dataset de validación (sin augmentación aleatoria)
dataset_val = DatasetParesSideBySide(
    directorio_raiz=str(DIR_DATOS / 'val'),
    direction='AtoB',
    modo='val',
    cache_en_ram=False,
)

print(f"Dataset val: {len(dataset_val)} pares")

# Extraer 4 muestras
N = 4
fig, axes = plt.subplots(N, 2, figsize=(8, 3 * N))
fig.suptitle('Pares tras el DataLoader (256×256, normalizados a [-1,1])', fontsize=12)

for i in range(N):
    tensor_a, tensor_b = dataset_val[i * 50]  # espaciar muestras

    # desnormalizar: [-1,1] → [0,1] para visualización
    img_a = desnormalizar(tensor_a).permute(1, 2, 0).numpy()
    img_b = desnormalizar(tensor_b).permute(1, 2, 0).numpy()

    axes[i, 0].imshow(img_a)
    axes[i, 0].set_title(f'A (satelital) — min={tensor_a.min():.2f}, max={tensor_a.max():.2f}', fontsize=8)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_b)
    axes[i, 1].set_title(f'B (mapa) — min={tensor_b.min():.2f}, max={tensor_b.max():.2f}', fontsize=8)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('results/exploracion_pares_procesados.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Estadísticas de Píxel

Verificamos que la normalización está correctamente aplicada. Con normalización `(x - 0.5) / 0.5 = 2x - 1`:
- Rango esperado: `[-1, 1]`
- Media esperada: `≈ 0` (si los píxeles tienen distribución aproximadamente simétrica)
- Std esperado: `≈ 0.5–0.8`

In [ ]:
dl_val = crear_dataloader(dataset_val, batch_size=16, modo='val', num_workers=0, shuffle=False)

batch_a_list, batch_b_list = [], []
for batch_a, batch_b in dl_val:
    batch_a_list.append(batch_a)
    batch_b_list.append(batch_b)
    if len(batch_a_list) >= 4:  # analizar primeros 64 pares
        break

todos_a = torch.cat(batch_a_list)
todos_b = torch.cat(batch_b_list)

print("Estadísticas de píxel (sobre primeros 64 pares):")
print(f"  Dominio A (satelital):")
print(f"    media = {todos_a.mean():.4f}  (esperado ≈ 0)")
print(f"    std   = {todos_a.std():.4f}  (esperado ≈ 0.5–0.8)")
print(f"    min   = {todos_a.min():.4f}  (esperado = -1.0)")
print(f"    max   = {todos_a.max():.4f}  (esperado =  1.0)")
print(f"\n  Dominio B (mapa OSM):")
print(f"    media = {todos_b.mean():.4f}")
print(f"    std   = {todos_b.std():.4f}")
print(f"    min   = {todos_b.min():.4f}")
print(f"    max   = {todos_b.max():.4f}")

# Histograma de intensidades
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, datos, titulo, color in [
    (axes[0], todos_a.numpy().flatten(), 'Dominio A — Satelital', 'steelblue'),
    (axes[1], todos_b.numpy().flatten(), 'Dominio B — Mapa OSM',  'darkorange'),
]:
    ax.hist(datos, bins=100, color=color, alpha=0.7, density=True)
    ax.axvline(datos.mean(), color='red', linestyle='--', label=f'Media = {datos.mean():.3f}')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('Valor de píxel (normalizado)')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)
    ax.set_xlim(-1.1, 1.1)

plt.tight_layout()
plt.savefig('results/histograma_pixeles.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n[OK] El rango [-1, 1] es compatible con la activación Tanh del generador.")

## Conclusión del EDA

**Observaciones clave:**

1. **Correspondencia perfecta**: cada par (A, B) muestra exactamente la misma área geográfica desde dos representaciones distintas. El modelo aprenderá a "traducir" entre ellas.

2. **Variedad de escenas**: el dataset incluye zonas urbanas densas, suburbios, parques y carreteras. Esta diversidad es esencial para generalización.

3. **Rango [-1, 1] verificado**: la normalización es correcta y compatible con `Tanh` en la salida del generador.

4. **Distribución asimétrica en dominio B**: los mapas OSM tienen muchos píxeles blancos (fondo) y pocas líneas de color. Esto puede hacer que la red aprenda el fondo más rápido que las líneas de carreteras.

Siguiente notebook: **02_architecture_demo.ipynb** — descripción matemática de la arquitectura.